# Preprocessing (generating subsets)

In [ ]:
import pandas as pd
import geopandas as gpd
import os

In [ ]:
path_geojson_rspo = "/data\\RSPO_Concessions_May_2025.geojson"
path_output = "/data\\subsets"

In [ ]:
gdf_geojson = gpd.read_file(path_geojson_rspo)

In [ ]:
gdf_geojson.columns

In [ ]:
provinces = [province for province in list(set(gdf_geojson['State'].to_list())) if province is not None]

In [ ]:
provinces = sorted(provinces)

In [ ]:
counter=1
for province in provinces:
    name = "RSPO_May_2025_subset" + str(counter) + ".geojson"
    gdf_subset = gdf_geojson[gdf_geojson['State'] == province]
    gdf_subset.to_file(os.path.join(path_output, name))
    counter+=1
gdf_geojson = None

# Formatting

### Country code in ISO 2 format
https://www.iban.com/country-codes

In [ ]:
file_example = os.path.join(path_output,"RSPO_May_2025_subset1.geojson")

In [ ]:
gdf_subset_example = gpd.read_file(file_example)
country_iso2_code = pd.read_csv("data\\ISO2_Code.csv")

In [ ]:
country = list(set(gdf_subset_example['Country'].to_list()))

In [ ]:
country_iso2_code[country_iso2_code['Country']==country[0]]['Alpha-2 code']

In [ ]:
eudr_columns = ['ProductionPlace', 'Area', 'ProducerCountry']

In [ ]:
gdf_subset_example.columns

In [ ]:
gdf_subset_example.rename({'Mill':'ProductionPlace', 'Country':'ProducerCountryName', 'AreaHa':'Area'}, axis='columns', inplace=True)

In [ ]:
len(set(gdf_subset_example['MemberNum'].to_list()))

In [ ]:
country_iso2_code.columns

In [ ]:
gdf_subset_example_filter = gdf_subset_example[['MemberNum','ProductionPlace', 'ProducerCountryName', 'Area']].copy()

In [ ]:
def iso2_code_func(column_dataframe):
    if column_dataframe['ProducerCountryName'] == country:
        return iso2_code

In [ ]:
for country in list(set(gdf_subset_example_filter['ProducerCountryName'].to_list())):
    iso2_code = country_iso2_code[country_iso2_code['Country']==country]['Alpha-2 code'].iloc[0]
    gdf_subset_example_filter['ProducerCountry'] = gdf_subset_example_filter.apply(iso2_code_func, axis=1)

In [ ]:
gdf_subset_example_filter

In [ ]:
subsets = os.listdir(path_output)
len(subsets)

In [ ]:
def iso2_code_func(row, iso_lookup_df):
    try:
        return iso_lookup_df.loc[iso_lookup_df['Country'] == row['ProducerCountryName'], 'Alpha-2 code'].iloc[0]
    except IndexError:
        return None

In [ ]:
producer_counter = [1]
def producer_name(row):
    if pd.isna(row['ProductionPlace']):
        name = f"ProductionPlace_{producer_counter[0]}"
        producer_counter[0] += 1
        return name
    return row['ProductionPlace']

In [ ]:
out_files_format = "C:\\Users\\Julio Collazos\\Documents\\GitHub\\30PublicationsChallenge\\eudr_geojson\\data\\RSPO_subsets_formated"
if not os.path.exists(out_files_format):
    os.mkdir(out_files_format)
for subset in subsets:
    subset_path = os.path.join(path_output, subset)
    gdf_temporary = gpd.read_file(subset_path)
    gdf_temporary.rename({'Mill':'ProductionPlace', 'Country':'ProducerCountryName', 'AreaHa':'Area'}, axis='columns', inplace=True)
    gdf_temporary_filter = gdf_temporary[['MemberNum','ProductionPlace', 'ProducerCountryName', 'Area', 'geometry']].copy()
    gdf_temporary_filter['ProducerCountry'] = gdf_temporary_filter.apply(lambda row: iso2_code_func(row, country_iso2_code), axis=1)
    gdf_temporary_filter['ProductionPlace'] = gdf_temporary_filter.apply(producer_name, axis=1)
    gdf_temporary_filter.to_file(os.path.join(out_files_format, subset), precision=6)

In [3]:
from datetime import date
print(date.today().strftime("%Y%m%d"))

20250902
